# `P06-E01` — Phân tích Kết quả Thực nghiệm Tổng hợp & Đóng Nghiên cứu (`CHANGE`)

**Dự án:** HOSTRA AI — Module `P06 (Sales Intelligence & Autonomous Storytelling)`  
**Phạm vi thực nghiệm:** `P06-E01` (Autonomous Storytelling không có bình luận người xem) — Phạm vi rút gọn được Tech Lead phê duyệt.  
**Nguyên tắc báo cáo bắt buộc:**
1. Tách biệt rạch ròi `Tier 1 (MockLLMProvider)`, `Pre-formal Pilot v1/v2` và `Formal Benchmark (10 runs, 317 Gemini API requests)`.
2. Tách biệt `AI-assisted non-blind evaluation (ChatGPT — Rubric v0.1)` khỏi `Human Blind Evaluation` & `GoldFactualLabel` (cả hai đang `PENDING`).
3. Không tự điền `0` cho `token_usage` bị `UNAVAILABLE`; không gộp mẫu số `topic-level` (Track 1) với `required-slot` (Track 2); không gán ghép cặp cross-scenario cho các cấu hình thiếu run tương ứng.

---

## 1. Bối cảnh Nghiên cứu & Mục tiêu (`RQ1–RQ5`, `H1–H4`)

Mục tiêu của `P06-E01` là kiểm chứng phương án kiến trúc **`Structured Story Planner + Dynamic Verified Generation (LangGraph)`** so với baseline **`Script-first`** trên 3 SKU thực phẩm đóng gói mô phỏng (`SKU-HONEY-01`, `SKU-TEA-02`, `SKU-MACADAMIA-03`), trả lời:
- **`RQ1` / `H1`:** Kể chuyện chủ động khi không bình luận — khả năng bao phủ `mandatory_topics` qua `PlaybackCompleted` ACK, mạch kể và giảm lặp ý mà không phát sinh vi phạm hard gate (`G4`).
- **`RQ5`:** Ảnh hưởng của cửa sổ lịch sử phát `Short-term Played History` (`W ∈ {5, 10, 20}`) và cơ chế viết lại giới hạn (`K=2`) lên độ phủ, lặp, độ trễ và token.
- *(Khảo sát mở rộng Track 2 — `EXP-W1..W4`, không phải RQ đăng ký ban đầu):* Đánh giá phản ứng ranh giới giữa `W` và `planned_lag` (`lag=6` & `lag=12`) dưới điều kiện stress `min_chunks_per_topic = 2`.


In [1]:
from __future__ import annotations

import hashlib
import json
import pathlib
import subprocess
import sys

import numpy as np
import pandas as pd

# Thêm notebooks/E01/ vào sys.path để nhập e01_analysis_utils bằng relative path
here = pathlib.Path.cwd().resolve()
for cand in [
    here,
    here / "E01",
    here / "notebooks" / "E01",
    here / "experiments" / "sales-intelligence" / "autonomous-storytelling" / "notebooks" / "E01",
]:
    if (cand / "e01_analysis_utils.py").is_file():
        if str(cand) not in sys.path:
            sys.path.insert(0, str(cand))
        break

import e01_analysis_utils as u

P06_ROOT = u.find_p06_root()
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
print("P06_ROOT:", P06_ROOT.name)
print("Provenance AI-assisted:", u.AI_EVAL_PROVENANCE)


P06_ROOT: autonomous-storytelling
Provenance AI-assisted: AI-assisted non-blind evaluation — ChatGPT — Rubric v0.1


---

## 2. Kiểm kê Toàn bộ Lượt chạy Thực nghiệm (`Formal Benchmark = 10 runs / 317 requests`)

Bảng dưới đây tách bạch **10 formal benchmark runs** (`317` API requests, `0 USD`, `0` HTTP 429, `8` failed calls đều là `HTTP 503`) khỏi các đợt **Pilot v1/v2** và **Offline Mock runs**.


In [2]:
pilot_df = u.load_pilot_and_mock_inventory(P06_ROOT)
runs_df = u.load_formal_runs_df(P06_ROOT)

print("=== PHẦN A: PILOT & MOCK RUNS (KHÔNG TÍNH VÀO 317 FORMAL REQUESTS) ===")
print(pilot_df[["phase_dir", "provider_tier", "model_id", "total_requests", "stage_b_runs", "included_in_317_formal"]].to_string(index=False))

print("\n=== PHẦN B: 10 FORMAL BENCHMARK RUNS ===")
inv_cols = ["batch_code", "full_label", "scenario", "min_chunks_per_topic", "requests_attempted", "failed_calls", "http_503_calls", "played_turns", "termination_reason"]
print(runs_df[inv_cols].to_string(index=False))
print(f"Tổng Formal Requests = {int(runs_df['requests_attempted'].sum())} | Tổng HTTP 503 failed calls = {int(runs_df['failed_calls'].sum())} | HTTP 429 = {int(runs_df['http_429_calls'].sum())} | Chi phí = {runs_df['cost_usd'].sum():.2f} USD")


=== PHẦN A: PILOT & MOCK RUNS (KHÔNG TÍNH VÀO 317 FORMAL REQUESTS) ===
                           phase_dir                provider_tier                model_id  total_requests  stage_b_runs  included_in_317_formal
         gemini_3.5_flash_lite_pilot pre_formal_pilot_gemini_live   gemini-3.5-flash-lite              25             2                   False
      gemini_3.5_flash_lite_pilot_v2 pre_formal_pilot_gemini_live   gemini-3.5-flash-lite              15             2                   False
post_fix_sweep_product_education_10m               unit_test_mock deterministic-mock-v0.1               0             6                   False
               offline_ext_v1_dryrun               unit_test_mock deterministic-mock-v0.1               0             6                   False

=== PHẦN B: 10 FORMAL BENCHMARK RUNS ===
batch_code        full_label                   scenario  min_chunks_per_topic  requests_attempted  failed_calls  http_503_calls  played_turns     termination_reason
  

---

## 3. Tải & Xác minh Bộ Dữ liệu (`e01_ext_v1` & `e01_formal_v1`)

Toàn bộ thí nghiệm dùng dữ liệu tổng hợp có kiểm soát (`is_synthetic = true`, `SYNTHETIC` only) cho 3 SKU: Mật ong hoa cà phê (`159,000đ`), Trà Shan Tuyết (`245,000đ`), Hạt mắc ca (`195,000đ`), kèm hồ sơ `OCOP: VERIFIED` mô phỏng.


In [3]:
for ds_name, rel_sku, rel_plan in [
    ("e01_ext_v1", "data/products/e01_ext_v1_verified_skus.json", "data/story_plans/e01_ext_v1_story_plans.json"),
    ("e01_formal_v1", "data/products/e01_formal_v1_verified_skus.json", "data/story_plans/e01_formal_v1_story_plans.json"),
]:
    sku_doc = json.loads((P06_ROOT / rel_sku).read_text(encoding="utf-8"))
    plan_doc = json.loads((P06_ROOT / rel_plan).read_text(encoding="utf-8"))
    plan_summary = {
        pid: (len([t for a in p["arcs"] for t in a["topics"]]), sum(1 for a in p["arcs"] for t in a["topics"] if t.get("is_mandatory")))
        for pid, p in plan_doc["plans"].items()
    }
    print(f"{ds_name}: is_synthetic={sku_doc['dataset_meta']['is_synthetic']} | skus={len(sku_doc['product_facts'])} | evidence_chunks={len(sku_doc['evidence_chunks'])} | plans(total,mand)={plan_summary}")


e01_ext_v1: is_synthetic=True | skus=3 | evidence_chunks=30 | plans(total,mand)={'product_education_10m_ext': (18, 12), 'brand_storytelling_10m_ext': (18, 12), 'sales_presentation_30m_ext': (24, 15), 'pilot_short_3sku_ext': (6, 6), 'window_sensitivity_stress_24t': (24, 24)}
e01_formal_v1: is_synthetic=True | skus=3 | evidence_chunks=36 | plans(total,mand)={'window_stress_12lag_3sku': (36, 36)}


---

## 4. Xác thực Toàn vẹn Hiện vật, SHA-256 & Cô lập `human_eval/`

Kiểm tra toàn bộ 6 tệp CSV đánh giá Track 2 nhập từ `E01_Track2_Final_Evaluation.zip` khớp SHA-256 từng byte, đồng thời bảo toàn nguyên vẹn toàn bộ thư mục `human_eval/` (cột người chấm giữ trống hoàn toàn).


In [4]:
expected_csv_hashes = {
    "track2a_small_window_lag6_s42/ai_assisted_eval/track2a_ai_assisted_quality.csv": "392c8d4e38fbe0fdfffa07259cdbc758ecb34b8abca1397cb32fa17ab176b2ea",
    "track2a_small_window_lag6_s42/ai_assisted_eval/track2a_ai_assisted_factual_semantic.csv": "3dee89839e7067bbe663d5545a73f154902f09698f732694c49727e42ffa80c4",
    "track2a_small_window_lag6_s42/ai_assisted_eval/track2a_ai_assisted_repetition.csv": "5f42a76f5b9a6565447e75f48d843564f18d0074d5178c4e1e4cf2c5f982b560",
    "track2b_large_window_lag12_s42/ai_assisted_eval/track2b_ai_assisted_quality.csv": "1f6e03d57be140d6b56d9b5cd8fa03c328733d268f96e512af576d6b2012fbe3",
    "track2b_large_window_lag12_s42/ai_assisted_eval/track2b_ai_assisted_factual_semantic.csv": "a5c02f5e18cabe60a702b88fa712a977865237574266130e9b1f7a34ca0e2863",
    "track2b_large_window_lag12_s42/ai_assisted_eval/track2b_ai_assisted_repetition.csv": "ce28170d44aeff4ccffc79527112745d56b3fa6b94e2679f3a0fe3eca2d9c77c",
}
fb_dir = P06_ROOT / "runs" / "P06-E01" / "formal_benchmark_v1"
for rel, exp in expected_csv_hashes.items():
    actual = hashlib.sha256((fb_dir / rel).read_bytes()).hexdigest()
    assert actual == exp, f"Hash mismatch: {rel}"
    print(f"SHA256 OK: {pathlib.Path(rel).name} -> {actual[:16]}...")

for bdir_name in ["track1_product_edu_s42", "track1_brand_storytelling_s42", "track2a_small_window_lag6_s42", "track2b_large_window_lag12_s42"]:
    df_he = pd.read_csv(fb_dir / bdir_name / "human_eval" / "blind_eval_sheet.csv", keep_default_na=False)
    assert (df_he["narrative_coherence"] == "").all(), f"Non-blank human_eval in {bdir_name}"
print("Human Evaluation Preservation Check: 4/4 formal batches blind_eval_sheet.csv giữ trống 100% (PENDING).")


SHA256 OK: track2a_ai_assisted_quality.csv -> 392c8d4e38fbe0fd...
SHA256 OK: track2a_ai_assisted_factual_semantic.csv -> 3dee89839e7067bb...
SHA256 OK: track2a_ai_assisted_repetition.csv -> 5f42a76f5b9a6565...
SHA256 OK: track2b_ai_assisted_quality.csv -> 1f6e03d57be140d6...
SHA256 OK: track2b_ai_assisted_factual_semantic.csv -> a5c02f5e18cabe60...
SHA256 OK: track2b_ai_assisted_repetition.csv -> ce28170d44aeff4c...
Human Evaluation Preservation Check: 4/4 formal batches blind_eval_sheet.csv giữ trống 100% (PENDING).


---

## 5. Phân tích Giai đoạn Pilot (`Pilot v1` vs `Pilot v2`)

Pilot v1 (`25` req) bộc lộ rò rỉ ID (`ev-...`) và vượt thời lượng `8.0s`; sau vòng sửa lỗi `R01–R06`, Pilot v2 (`15` req, `0` HTTP 503) xác nhận LangGraph `W=5` đạt `6/6` topic & claim coverage với `0` wasted calls và `0` leaks, đủ điều kiện mở khóa Formal Benchmark.


In [5]:
for pv in ["gemini_3.5_flash_lite_pilot", "gemini_3.5_flash_lite_pilot_v2"]:
    pdir = P06_ROOT / "runs" / "P06-E01" / pv
    comp = json.loads((pdir / "comparison.json").read_text(encoding="utf-8"))
    print(f"=== {pv} ===")
    for row in comp["rows"]:
        print(f"  method={row['method']} W={row['history_window']} mand_cov={row['mandatory_topic_coverage']} exact_text={row['exact_normalized_text_replay_count']} rw={row['rewrite_rate']:.3f} calls={row['total_model_calls']} gate={row['overall_gate_status']}")


=== gemini_3.5_flash_lite_pilot ===
  method=script_first_upfront_llm W=None mand_cov=0.8333333333333334 exact_text=0 rw=0.000 calls=6 gate=NOT_EVALUATED
  method=langgraph_structured_story W=5 mand_cov=1.0 exact_text=2 rw=0.444 calls=16 gate=NOT_EVALUATED
=== gemini_3.5_flash_lite_pilot_v2 ===
  method=script_first_upfront_llm W=None mand_cov=0.8333333333333334 exact_text=0 rw=0.000 calls=6 gate=NOT_EVALUATED
  method=langgraph_structured_story W=5 mand_cov=1.0 exact_text=0 rw=0.000 calls=6 gate=NOT_EVALUATED


---

## 6. Phân tích Natural Track (`Batch 1A Product Edu` & `Batch 1B Brand Storytelling`)

Trên cả 2 kịch bản tự nhiên (`min_chunks_per_topic = 1`), **`LangGraph` đạt `100%` (`12/12` mandatory & `18/18` planned topics qua playback ACK)**, trong khi `Script-first` hụt xuống `58.33%` (1A) và `75.00%` (1B) do không có cơ chế bounded rewrite khi bản nháp đầu vượt giới hạn thời lượng hoặc rớt P05. Khi so sánh ghép cặp cross-scenario, chỉ đối chiếu **`Script-first` vs `LangGraph W=5`** (vì 1B chỉ chạy cặp này).


In [6]:
nat_df = runs_df[runs_df["batch_code"].isin(["1A", "1B"])].copy()
cols_nat = ["batch_code", "scenario_label", "config_name", "termination_reason", "completed_mandatory_topics", "mandatory_topics_count", "mandatory_topic_coverage", "completed_planned_topics", "planned_topics_count", "lexical_repetition_rate", "exact_normalized_text_replay_count", "rewrite_rate", "fallback_rate"]
print(nat_df[cols_nat].to_string(index=False))

paired = nat_df[nat_df["config_name"].isin(["Script-first", "LangGraph W=5"])]
print("\n=== CROSS-SCENARIO PAIRED SUMMARY (Script-first vs LangGraph W=5 ONLY) ===")
print(paired[["batch_code", "scenario_label", "config_name", "mandatory_topic_coverage", "planned_topic_coverage", "simulated_played_duration_sec"]].to_string(index=False))


batch_code     scenario_label    config_name termination_reason  completed_mandatory_topics  mandatory_topics_count  mandatory_topic_coverage  completed_planned_topics  planned_topics_count  lexical_repetition_rate  exact_normalized_text_replay_count  rewrite_rate  fallback_rate
        1A  Product Education   Script-first  CONTENT_EXHAUSTED                           7                      12                    0.5833                        10                    18                      0.0                                   0        0.0000         0.2778
        1A  Product Education  LangGraph W=5   EARLY_COMPLETION                          12                      12                    1.0000                        18                    18                      0.0                                   0        0.2222         0.0000
        1A  Product Education LangGraph W=10   EARLY_COMPLETION                          12                      12                    1.0000                   

---

## 7. Phân tích Độ nhạy Cửa sổ Lịch sử (`Track 2A lag=6` & `Track 2B lag=12` + Mock vs Live)

Trong điều kiện stress (`min_chunks_per_topic = 2`, mỗi topic chỉ có `1` `allowed_claim_id`), hành vi chia rõ theo quan hệ giữa `W` và `actual_lag`:
- `W < lag` (`2A W=5`, `2B W=10`): hoàn tất `100%` chủ đề nhưng bỏ lọt lặp ngoài cửa sổ (`lexical_repetition_rate` `28.57%–32.39%` và `exact_normalized_text_replay` = `2` và `1`).
- `W > lag` (`2A W=10`, `2B W=20`): bế tắc `REPEATED_FACTS` ngay chu kỳ quay lại đầu tiên của `SKU-HONEY-01`, cạn fallback rồi dừng **`MAX_CONSECUTIVE_ABORTS` (`0%` hoàn tất `>= 2` chunks)**.
- *Ghi chú phương pháp luận:* Mock `unit_test` cho thấy `18/18` (2A `W=10`) và `35/36` (2B `W=20`), lệch rất lớn so với Gemini Live (`0/18` và `0/36`).


In [7]:
t2_df = runs_df[runs_df["batch_code"].isin(["2A", "2B"])].copy()
t2_cols = ["batch_code", "planned_lag", "config_name", "termination_reason", "completed_planned_topics", "planned_topics_count", "outside_window_exact_text_replay_count", "lexical_repetition_rate", "detector_none", "detector_lexical_dup", "detector_repeated_facts", "fallback_count", "safe_abort_count"]
print(t2_df[t2_cols].to_string(index=False))

mvl_df = u.load_track2_mock_vs_live_df(P06_ROOT)
print("\n=== MOCK VS LIVE COMPARISON (TÁCH TẦNG) ===")
print(mvl_df[["batch_code", "planned_lag", "history_window", "provider_tier", "total_calls", "facts_rejects", "safe_abort_count", "completed_topics_ge2", "termination_reason"]].to_string(index=False))


batch_code planned_lag    config_name     termination_reason  completed_planned_topics  planned_topics_count outside_window_exact_text_replay_count  lexical_repetition_rate  detector_none  detector_lexical_dup  detector_repeated_facts  fallback_count  safe_abort_count
        2A           6  LangGraph W=5       EARLY_COMPLETION                        18                    18                                      2                   0.2857             36                     0                        0               0                 0
        2A           6 LangGraph W=10 MAX_CONSECUTIVE_ABORTS                         0                    18                                      0                   0.0000              6                     6                        8               2                 3
        2B          12 LangGraph W=10       EARLY_COMPLETION                        36                    36                                      1                   0.3239             72      

---

## 8. Phân bố `Actual Lag` & Độ chồng chéo N-gram qua Ngưỡng Lag

Trên cả hai phiên chạy hoàn thành, phân bố `actual_lag` của các lượt phát lại đều chính xác tuyệt đối `{6: 18}` (T2A `W=5`) và `{12: 36}` (T2B `W=10`), cho thấy 100% lượt quay lại rơi ngoài `W`. Ngay khi cửa sổ đo lường mở rộng qua ngưỡng `lag`, `windowed_ngram3_jaccard_mean` bật tăng `~73.4×` (T2A `w5→w10`) và `~18.8×` (T2B `w10→w20`).


In [8]:
lag_sum = u.load_track2_lag_summary(P06_ROOT)
for bcode, info in lag_sum.items():
    print(f"Track {bcode}: pass W={info['pass_w']} played_revisits={info['played_revisits_count']} dist={info['played_lag_dist']} | fail W={info['fail_w']} rejected_attempts={info['fail_attempted_revisits_count']} in_window={info['fail_in_window_attempts']} dist={info['fail_attempted_lag_dist']}")
print("\nWindowed N-gram3 Jaccard (passing stress runs):")
print(t2_df[t2_df["termination_reason"] == "EARLY_COMPLETION"][["full_label", "win_ngram3_w5", "win_ngram3_w10", "win_ngram3_w20"]].to_string(index=False))


Track 2A: pass W=5 played_revisits=18 dist={6: 18} | fail W=10 rejected_attempts=14 in_window=14 dist={5: 3, 6: 2, 7: 6, 8: 3}
Track 2B: pass W=10 played_revisits=36 dist={12: 36} | fail W=20 rejected_attempts=13 in_window=13 dist={11: 1, 12: 6, 13: 3, 14: 3}

Windowed N-gram3 Jaccard (passing stress runs):
       full_label  win_ngram3_w5  win_ngram3_w10  win_ngram3_w20
 2A LangGraph W=5         0.0028          0.2031          0.2031
2B LangGraph W=10         0.0094          0.0131          0.2455


---

## 9. Độ phủ Cấu trúc vs Độ phủ Ngữ nghĩa (`Topic-level` vs `Required-Slot`, Tách Mẫu Số)

Bằng chứng định lượng cho thấy **`playback-confirmed topic coverage = 100%` KHÔNG đồng nghĩa `semantic FULL = 100%`**:
- Trong **Natural Track (`m=1`)**, `1A LangGraph W=5` đạt `100%` playback nhưng chỉ `7/18` (`38.9%`) đạt `FULL` ngữ nghĩa (`11/18 PARTIAL`, thiếu đơn vị hoặc ý phụ để vừa trần từ); `1B W=5` cải thiện lên `12/18` (`66.7% FULL`, `6 PARTIAL`).
- Trong **Track 2 (`m=2`, đơn vị `required slot`)**, `54` dòng thiếu đầu ra tương ứng **`90` missing required slots** (`30` ở 2A `W=10`, `60` ở 2B `W=20`, vì mỗi `MISSING_TOPIC` đại diện `2` required slots). Số topic đạt `FULL` cả 2 chunks chỉ đạt `3/18` (2A `W=5`) và `28/36` (2B `W=10`).


In [9]:
eval_dfs = u.load_ai_assisted_eval_dfs(P06_ROOT, runs_df)
sem_df = eval_dfs["semantic"]
print(sem_df[["batch_code", "full_label", "coverage_unit", "all_denominator", "all_FULL", "all_PARTIAL", "all_MISS", "mandatory_FULL", "mandatory_PARTIAL", "mandatory_MISS", "missing_output_rows", "missing_required_slots", "topics_both_chunks_FULL"]].to_string(index=False))
print("Track 2 total missing_output_rows =", int(sem_df[sem_df["batch_code"].isin(["2A", "2B"])]["missing_output_rows"].sum()), "| total missing_required_slots =", int(sem_df[sem_df["batch_code"].isin(["2A", "2B"])]["missing_required_slots"].sum()))


batch_code        full_label    coverage_unit  all_denominator  all_FULL  all_PARTIAL  all_MISS  mandatory_FULL  mandatory_PARTIAL  mandatory_MISS  missing_output_rows  missing_required_slots topics_both_chunks_FULL
        1A 1A LangGraph W=20   topic_level_m1               18         5           13         0               3                  9               0                    0                       0                     NaN
        1A 1A LangGraph W=10   topic_level_m1               18         7           11         0               3                  9               0                    0                       0                     NaN
        1A   1A Script-first   topic_level_m1               18         4            6         8               2                  5               5                    3                       3                     NaN
        1A  1A LangGraph W=5   topic_level_m1               18         7           11         0               3                  9      

---

## 10. Kiểm toán Factual Correctness & Evidence Support (AI-Assisted ChatGPT)

Trên **mẫu số chỉ gồm `played turns`** (gồm fallback đã phát, loại trừ missing output; `UNSURE` không tính `PASS`):
- **Natural Track (`1A` + `1B`, `110` played turns):** Factual `94 PASS` (`85.45%`), `5 FAIL`, `11 UNSURE`; Evidence Support `84 PASS` (`76.36%`); Metadata leak `0/110`.
- **Track 2 (`2A` + `2B`, `130` played turns):** Factual **`104 PASS` (`80.00%`), `11 FAIL` (`8.46%`), `15 UNSURE` (`11.54%`)**; Evidence Support **`100 PASS` (`76.92%`, `30 FAIL`)**; Metadata leak **`0/130`**.
- **`11` Track 2 Factual FAILs** phân nhóm: `5` ca mất phạm vi *hồ sơ mô phỏng* / khẳng định *an toàn tuyệt đối* ở OCOP (`*-10`), `4` ca thiếu điều kiện *khi nhận hàng* / *khi mở hộp* trong đổi trả (`*-09`), và `2` ca biến dạng / thiếu cảnh báo an toàn (`2A S2 t25` `macadamia-safety`, `2B S2 t42` `tea-07`).


In [10]:
fac_df = eval_dfs["factual"]
print(fac_df[["batch_code", "full_label", "played_turns_denominator", "missing_output_rows_excluded", "factual_PASS", "factual_FAIL", "factual_UNSURE", "factual_PASS_pct", "evidence_PASS", "evidence_FAIL", "evidence_PASS_pct", "metadata_leak_YES"]].to_string(index=False))

t2_fac = fac_df[fac_df["batch_code"].isin(["2A", "2B"])]
tot_p = int(t2_fac["played_turns_denominator"].sum())
print(f"\nTrack 2 Total ({tot_p} played): Factual PASS={int(t2_fac['factual_PASS'].sum())} ({100*t2_fac['factual_PASS'].sum()/tot_p:.2f}%), FAIL={int(t2_fac['factual_FAIL'].sum())} ({100*t2_fac['factual_FAIL'].sum()/tot_p:.2f}%), UNSURE={int(t2_fac['factual_UNSURE'].sum())} ({100*t2_fac['factual_UNSURE'].sum()/tot_p:.2f}%), Evidence PASS={int(t2_fac['evidence_PASS'].sum())} ({100*t2_fac['evidence_PASS'].sum()/tot_p:.2f}%)")


batch_code        full_label  played_turns_denominator  missing_output_rows_excluded  factual_PASS  factual_FAIL  factual_UNSURE  factual_PASS_pct  evidence_PASS  evidence_FAIL  evidence_PASS_pct  metadata_leak_YES
        1A 1A LangGraph W=20                        20                             0            17             0               3             85.00             16              4              80.00                  0
        1A 1A LangGraph W=10                        21                             0            18             1               2             85.71             16              5              76.19                  0
        1A   1A Script-first                        15                             3            14             0               1             93.33             12              3              80.00                  0
        1A  1A LangGraph W=5                        18                             0            14             0               4            

---

## 11. Phân tích Lặp Ngữ nghĩa Trên Các Chủ đề Quay lại (`108` Planned Revisits)

Trong `108` cơ hội quay lại ở Track 2, có `54` lượt phát thành công (`2A W=5` = `18`, `2B W=10` = `36`) và `54 NO_OUTPUT` (`2A W=10` = `18`, `2B W=20` = `36`). Trong `54` lượt thực sự phát:
- **`35 / 54` (`64.81%`)** là lặp nguyên văn / gần nguyên văn (`19 EXACT_REPETITION`) hoặc lặp ý không thêm giá trị (`16 UNWANTED_SEMANTIC_REPETITION`).
- Chỉ **`13 / 54` (`24.07%`)** đạt `LEGITIMATE_CONTINUATION` (bổ sung chi tiết mới từ kho tri thức).
- Trong `19` ca gắn nhãn `EXACT_REPETITION` theo reviewer guide, chỉ **`3`** cặp trùng nguyên văn tuyệt đối (`2` ở `2A W=5`, `1` ở `2B W=10`), còn **`16`** cặp là gần nguyên văn — nên **không** đồng nhất `EXACT_REPETITION` với metric tự động `exact_normalized_text_replay`.


In [11]:
rep_df = eval_dfs["repetition"]
print(rep_df[["batch_code", "full_label", "planned_revisits", "played_revisits", "EXACT_REPETITION", "exact_verbatim_pairs", "UNWANTED_SEMANTIC_REPETITION", "LEGITIMATE_CONTINUATION", "DELIBERATE_RECURRENCE", "UNSURE", "NO_OUTPUT", "unwanted_or_exact_played_pct"]].to_string(index=False))

played_rows = rep_df[rep_df["played_revisits"] > 0]
p_sum = int(played_rows["played_revisits"].sum())
exact_n = int(played_rows["EXACT_REPETITION"].sum())
unw_n = int(played_rows["UNWANTED_SEMANTIC_REPETITION"].sum())
leg_n = int(played_rows["LEGITIMATE_CONTINUATION"].sum())
del_n = int(played_rows["DELIBERATE_RECURRENCE"].sum())
uns_n = int(played_rows["UNSURE"].sum())
verb_n = int(played_rows["exact_verbatim_pairs"].sum())
print(f"Played Revisits Total={p_sum}: EXACT={exact_n} (verbatim={verb_n}, near={exact_n-verb_n}), UNWANTED={unw_n}, EXACT+UNWANTED={exact_n+unw_n}/{p_sum} ({100*(exact_n+unw_n)/p_sum:.2f}%), LEGITIMATE={leg_n}/{p_sum} ({100*leg_n/p_sum:.2f}%), DELIBERATE={del_n}, UNSURE={uns_n}")


batch_code        full_label  planned_revisits  played_revisits  EXACT_REPETITION  exact_verbatim_pairs  UNWANTED_SEMANTIC_REPETITION  LEGITIMATE_CONTINUATION  DELIBERATE_RECURRENCE  UNSURE  NO_OUTPUT  unwanted_or_exact_played_pct
        2A 2A LangGraph W=10                18                0                 0                     0                             0                        0                      0       0         18                           NaN
        2A  2A LangGraph W=5                18               18                 5                     2                             4                        7                      2       0          0                         50.00
        2B 2B LangGraph W=20                36                0                 0                     0                             0                        0                      0       0         36                           NaN
        2B 2B LangGraph W=10                36               36             

---

## 12. Độ tin cậy, Rewrite, Fallback & Safe Abort

Trong tổng `317` formal requests, có đúng **`8` failed calls** đều là lỗi `HTTP 503 UNAVAILABLE` từ Google AI Studio (`1A W=10`: `3`, `1A W=20`: `1`, `1B Script-first`: `1`, `1B W=5`: `1`, `2A W=10`: `1`, `2B W=20`: `1`), và **`0` HTTP 429**. Tuy nhiên ở nhóm chạy hết kịch bản (`W < lag` hoặc Track 1), **tỷ lệ rewrite thuần logic (`OVER_MAX_DURATION_SEC` vượt 20 từ / 8.0s)** vẫn lên tới `13.0%–35.0%` wasted calls.


In [12]:
rel_cols = ["batch_code", "full_label", "requests_attempted", "failed_calls", "wasted_calls", "wasted_calls_pct", "rewrite_count", "rewrite_rate", "fallback_count", "fallback_rate", "safe_abort_count", "termination_reason"]
print(runs_df[rel_cols].to_string(index=False))


batch_code        full_label  requests_attempted  failed_calls  wasted_calls  wasted_calls_pct  rewrite_count  rewrite_rate  fallback_count  fallback_rate  safe_abort_count     termination_reason
        1A   1A Script-first                  18             0             8             44.44              0        0.0000               5         0.2778                 3      CONTENT_EXHAUSTED
        1A  1A LangGraph W=5                  22             0             4             18.18              4        0.2222               0         0.0000                 0       EARLY_COMPLETION
        1A 1A LangGraph W=10                  27             3             9             33.33              6        0.1905               3         0.1429                 0       EARLY_COMPLETION
        1A 1A LangGraph W=20                  27             1             9             33.33              7        0.2500               2         0.1000                 0       EARLY_COMPLETION
        1B   1B Scri

---

## 13. Giới hạn Hạch toán Token & Độ trễ Provider

- Các phiên không có `HTTP 503` đạt `token_usage_source = MEASURED` (`4/10` runs: `1A Script-first`, `1A W=5`, `2A W=5`, `2B W=10`).
- `6/10` runs còn lại mang nhãn `UNAVAILABLE` ở tổng hợp vì có `1–3` failed calls; chúng tôi báo cáo riêng **`partial_measured_tokens`** trên các calls thành công và **tuyệt đối không điền `0`** cho `input_tokens`/`output_tokens` tổng hợp.
- `provider_latency_p50_ms` (`~5.90s–6.56s`) **bao gồm pacing phía client (`min_request_interval_sec = 6.0`)** nên không đại diện cho pure inference latency của Gemini.


In [13]:
tok_cols = ["batch_code", "full_label", "token_usage_source", "input_tokens", "output_tokens", "wasted_tokens", "partial_measured_input_tokens", "partial_measured_output_tokens", "provider_latency_p50_ms", "provider_latency_p95_ms"]
print(runs_df[tok_cols].to_string(index=False))


batch_code        full_label token_usage_source  input_tokens  output_tokens  wasted_tokens  partial_measured_input_tokens  partial_measured_output_tokens  provider_latency_p50_ms  provider_latency_p95_ms
        1A   1A Script-first           MEASURED       38845.0         3325.0        18709.0                          38845                            3325                   5967.2                   6238.7
        1A  1A LangGraph W=5           MEASURED       65225.0         2631.0        12324.0                          65225                            2631                   5953.2                   7387.2
        1A 1A LangGraph W=10        UNAVAILABLE           NaN            NaN            NaN                          80689                            3002                   5999.3                   8141.4
        1A 1A LangGraph W=20        UNAVAILABLE           NaN            NaN            NaN                          90419                            3110                   5958.1 

---

## 14. Thời lượng Mô phỏng (`Duration Fulfillment` vs Mục tiêu 10–30 Phút)

Sử dụng bộ đếm giả lập `VirtualClock` (`words / 2.6` giây), phiên dài nhất (`Track 2B W=10`, `72` lượt phát lặp 2 vòng trên `36` chủ đề) chỉ đạt **`489.996s` (`81.67%` của `600s`)**. Natural Track (`18` chủ đề) chỉ đạt `105.385s–145.000s` (`17.6%–24.2%` của `600s`). **Chưa chứng minh được mục tiêu 10–30 phút và chưa đo trên TTS audio thực tế.**


In [14]:
print(runs_df[["batch_code", "full_label", "played_turns", "simulated_played_duration_sec", "target_duration_sec", "duration_fulfillment", "termination_reason"]].to_string(index=False))


batch_code        full_label  played_turns  simulated_played_duration_sec  target_duration_sec  duration_fulfillment     termination_reason
        1A   1A Script-first            15                        105.385                600.0                0.1756      CONTENT_EXHAUSTED
        1A  1A LangGraph W=5            18                        124.231                600.0                0.2071       EARLY_COMPLETION
        1A 1A LangGraph W=10            21                        145.000                600.0                0.2417       EARLY_COMPLETION
        1A 1A LangGraph W=20            20                        134.615                600.0                0.2244       EARLY_COMPLETION
        1B   1B Script-first            17                        118.077                600.0                0.1968      CONTENT_EXHAUSTED
        1B  1B LangGraph W=5            19                        124.229                600.0                0.2070       EARLY_COMPLETION
        2A  2A LangG

---

## 15. Đánh giá Acceptance Criteria (`14` Tiêu chí) & Xuất Toàn bộ Bảng / Biểu đồ / Index

Xuất đồng bộ toàn bộ `12` bảng CSV (`reports/tables/E01/`), `8` biểu đồ PNG (`reports/figures/E01/`) và `reports/E01/e01_artifact_index.json`.


In [15]:
lag_summary = u.load_track2_lag_summary(P06_ROOT)
table_paths = u.export_all_tables(P06_ROOT, runs_df, pilot_df, eval_dfs, mvl_df)
figure_paths = u.generate_all_figures(P06_ROOT, runs_df, eval_dfs, lag_summary)

git_branch = subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=P06_ROOT, text=True).strip()
git_head = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=P06_ROOT, text=True).strip()
index_path = u.export_artifact_index(P06_ROOT, runs_df, eval_dfs, table_paths, figure_paths, {"branch": git_branch, "head_before_closure_commit": git_head})

ac_df = pd.read_csv(table_paths["11_acceptance_criteria_assessment"])
print("=== BẢNG 11: ACCEPTANCE CRITERIA (14 TIÊU CHÍ) ===")
print(ac_df[["criterion_id", "criterion", "formal_status", "observed_evidence"]].to_string(index=False))
print(f"\nExported {len(table_paths)} CSV tables, {len(figure_paths)} PNG figures, and {index_path.name}.")


=== BẢNG 11: ACCEPTANCE CRITERIA (14 TIÊU CHÍ) ===
 criterion_id                                                                                 criterion formal_status                                                                                                                                                                                                                          observed_evidence
            1 Structural invariants (exact_chunk_replay=0, metadata_leak=0, p05_reject=0, retry_viol=0)          PASS                                                                                                                10/10 formal runs structural_hard_gates_status = PASS; exact_chunk_replay=0; retry_viol=0; metadata_leak=0/240 played turns
            2                                     Natural Track mandatory coverage (playback-confirmed)          PASS                                                                                                                              

---

## 16. Kết luận Nghiên cứu & Chốt `W=5` (`CHANGE`)

- **Quyết định Cuối cùng:** **`CHANGE`** — Giữ kiến trúc cốt lõi (`Structured Story Planner + LangGraph + Dynamic Verified Generation + Playback-confirmed progress + P05 Adapter + Bounded Retry`), nhưng phải sửa lỗi semantic safety/rewrite/revisit policy ở `E04/E02/P05` trước khi nghiệm thu MVP.
- **Chốt cấu hình tạm thời cho Natural Storytelling:** **`W = 5`** (`provisional default`, không tuyên bố tối ưu toàn cục); `W = 10` và `W = 20` giữ vai trò cấu hình đối chứng/stress.
- **Đề xuất nghiên cứu tiếp theo (chưa validated, chưa triển khai trong E01):** Kiến trúc memory 3 tầng (`Session-level exact text hash memory` + `Short-term context history` + `Claim/topic/angle progress memory`).


In [16]:
print("Final Decision: CHANGE")
print("Provisional Default (Natural Track): W=5")
print("Track 2 Comparison Configs: W=10 (research comparison), W=20 (stress comparison)")
print("Next Experiment Handoff: P06-E04 (Bounded Rewrite, Qualifier Preservation & Recovery/Abort Policy)")


Final Decision: CHANGE
Provisional Default (Natural Track): W=5
Track 2 Comparison Configs: W=10 (research comparison), W=20 (stress comparison)
Next Experiment Handoff: P06-E04 (Bounded Rewrite, Qualifier Preservation & Recovery/Abort Policy)


---

## 17. Đe dọa tới Tính Hợp lệ (`Threats to Validity`)

1. **`Human Blind Evaluation`, `Human Calibration` & `GoldFactualLabel` đều `PENDING`:** Mọi nhãn chất lượng mềm, semantic alignment và factual/evidence đều đến từ **`AI-assisted non-blind evaluation (ChatGPT — Rubric v0.1)`** — không thay thế được người chấm độc lập; trạng thái cổng chính thức giữ nguyên **`NOT_EVALUATED`**.
2. **`n = 1` mỗi cấu hình & không tổng hợp p-value:** Mỗi run chỉ chạy 1 hạt giống (`seed=42`) trên Free Tier, chỉ mô tả xu hướng định lượng & cơ chế.
3. **`HTTP 503` đơn lẻ (`8/317` calls):** Xuất hiện ở `6/10` runs làm `token_usage` chuyển `UNAVAILABLE` ở tổng hợp và tham gia rút ngắn 1 bước abort ở Track 2A `W=10` & Track 2B `W=20` (dù bế tắc `REPEATED_FACTS` & cạn fallback đã xảy ra trước lỗi `503`).
4. **Dữ liệu tổng hợp & `VirtualClock` proxy:** Toàn bộ tri thức là `SYNTHETIC` và thời lượng ước lượng theo `words/2.6` chưa qua TTS (`P02`) hay livestream thực tế (`P04`).


In [17]:
from PIL import Image
for name, fp in sorted(figure_paths.items()):
    with Image.open(fp) as im:
        print(f"Figure OK: {fp.name} | size={im.size} | bytes={fp.stat().st_size}")
for name, tp in sorted(table_paths.items()):
    assert tp.is_file() and tp.stat().st_size > 100
print("ALL TABLES, FIGURES & ARTIFACT INDEX VERIFIED SUCCESSFULLY.")


Figure OK: fig01_natural_mandatory_coverage.png | size=(1600, 960) | bytes=61639
Figure OK: fig02_structural_vs_semantic_coverage.png | size=(1600, 960) | bytes=58890
Figure OK: fig03_ai_assisted_storytelling_quality.png | size=(1600, 960) | bytes=54120
Figure OK: fig04_window_completion_vs_repetition_tradeoff.png | size=(1600, 960) | bytes=60818
Figure OK: fig05_actual_lag_and_window_membership.png | size=(1600, 960) | bytes=71432
Figure OK: fig06_semantic_repetition_distribution.png | size=(1600, 960) | bytes=70163
Figure OK: fig07_factual_and_evidence_assessment.png | size=(1600, 960) | bytes=53522
Figure OK: fig08_reliability_waste_and_duration.png | size=(1600, 960) | bytes=65758
ALL TABLES, FIGURES & ARTIFACT INDEX VERIFIED SUCCESSFULLY.
